In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
%cd /content/drive/MyDrive/ASR

/content/drive/MyDrive/ASR


# Fine-Tune Whisper For Multilingual ASR with 🤗 Transformers

We can verify that we've been assigned a GPU and view its specifications:

In [3]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

/bin/bash: line 1: nvidia-smi: command not found


In [4]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load Dataset

In [5]:
"""
1_preprocess.py
Preprocess LibriSpeech (clean, train.100 subset) for Whisper Small fine-tuning.
Saves processed dataset to reuse later.
"""

from datasets import load_dataset, DatasetDict, Audio
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
import os

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID      = "openai/whisper-small"
LANGUAGE      = "english"
TASK          = "transcribe"
SAMPLING_RATE = 16_000
SAVE_DIR      = "./whisper_dataset"

# Correct splits (FIXED)
TRAIN_SPLIT = "train.100[:200]"
VALID_SPLIT = "validation[:50]"
# ──────────────────────────────────────────────────────────────────────────────

print("Loading feature extractor, tokenizer, and processor …")

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(
    MODEL_ID,
    language=LANGUAGE,
    task=TASK
)
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language=LANGUAGE,
    task=TASK
)


def load_splits() -> DatasetDict:
    print("Downloading LibriSpeech splits …")

    train = load_dataset(
        "librispeech_asr",
        "clean",
        split=TRAIN_SPLIT
    )

    valid = load_dataset(
        "librispeech_asr",
        "clean",
        split=VALID_SPLIT
    )

    return DatasetDict({
        "train": train,
        "validation": valid
    })


def prepare_features(batch):
    """
    Audio → log-Mel features + text → token ids
    """

    audio = batch["audio"]

    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = tokenizer(batch["text"]).input_ids

    batch["input_length"] = len(audio["array"]) / audio["sampling_rate"]

    return batch


def is_short_enough(batch, max_seconds: float = 30.0):
    return batch["input_length"] <= max_seconds


def main():
    ds = load_splits()

    # Resample audio
    print("Resampling audio to 16 kHz …")
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

    # Feature extraction
    print("Extracting features + tokenizing …")
    ds = ds.map(
        prepare_features,
        remove_columns=ds["train"].column_names,
        num_proc=1
    )

    # Filter long audio (>30s)
    before = {k: len(v) for k, v in ds.items()}
    ds = ds.filter(is_short_enough)
    after = {k: len(v) for k, v in ds.items()}

    for split in ds:
        removed = before[split] - after[split]
        print(f"[{split}] kept {after[split]} / {before[split]} (removed {removed})")

    # Save dataset
    print(f"Saving dataset to {SAVE_DIR} …")
    os.makedirs(SAVE_DIR, exist_ok=True)
    ds.save_to_disk(SAVE_DIR)

    # Save processor
    processor.save_pretrained(SAVE_DIR)

    print("Done. Dataset ready.")
    print(ds)


if __name__ == "__main__":
    main()

Loading feature extractor, tokenizer, and processor …


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Filter:   0%|          | 0/50 [00:00<?, ? examples/s]

[train] kept 200 / 200 (removed 0)
[validation] kept 50 / 50 (removed 0)
Saving dataset to ./whisper_dataset …


Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Done. Dataset ready.
DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels', 'input_length'],
        num_rows: 200
    })
    validation: Dataset({
        features: ['input_features', 'labels', 'input_length'],
        num_rows: 50
    })
})


In [5]:
!pip install evaluate>=0.4.1
!pip install jiwer>=3.0.3

In [6]:
!pip install jiwer>=3.0.3

In [ ]:
"""
2_finetune.py
Fine-tune Whisper Small on the preprocessed LibriSpeech dataset.
Evaluates WER on the validation split after every epoch.
"""

import os
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_from_disk
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import evaluate

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID     = "openai/whisper-small"
DATASET_DIR  = "./whisper_dataset"
OUTPUT_DIR   = "./whisper-small-librispeech"
LANGUAGE     = "english"
TASK         = "transcribe"

# Training hyper-params (tiny values for demo; scale up for real training)
BATCH_SIZE       = 4      # per-device
GRAD_ACCUM_STEPS = 2      # effective batch = BATCH_SIZE * GRAD_ACCUM_STEPS
LEARNING_RATE    = 1e-5
WARMUP_STEPS     = 50
MAX_STEPS        = 200    # set -1 to train for full epochs instead
NUM_EPOCHS       = 3      # used only when MAX_STEPS == -1
FP16             = torch.cuda.is_available()
# ──────────────────────────────────────────────────────────────────────────────


# ── Data collator ─────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad input features (log-Mel spectrograms) to the same length
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad label sequences; replace padding token id with -100 so loss ignores them
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Strip BOS token if present (Whisper adds it during generation, not training)
        bos_token_id = self.processor.tokenizer.bos_token_id
        if (labels[:, 0] == bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch
# ──────────────────────────────────────────────────────────────────────────────


def compute_metrics_fn(processor):
    """Return a compute_metrics function bound to the processor."""
    wer_metric = evaluate.load("wer")
    tokenizer  = processor.tokenizer

    def compute_metrics(pred):
        pred_ids   = pred.predictions
        label_ids  = pred.label_ids

        # Replace -100 back to pad token id so decode doesn't fail
        label_ids[label_ids == -100] = tokenizer.pad_token_id

        pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        # Normalise: lower-case, strip leading/trailing spaces
        pred_str  = [p.lower().strip() for p in pred_str]
        label_str = [l.lower().strip() for l in label_str]

        wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
        return {"wer": round(wer, 2)}

    return compute_metrics


def main():
    # ── Load dataset & processor ──────────────────────────────────────────────
    print("Loading preprocessed dataset …")
    ds        = load_from_disk(DATASET_DIR)
    processor = WhisperProcessor.from_pretrained(DATASET_DIR, language=LANGUAGE, task=TASK)

    print(f"  Train size : {len(ds['train'])}")
    print(f"  Valid size : {len(ds['validation'])}")

    # ── Model ─────────────────────────────────────────────────────────────────
    print(f"Loading model '{MODEL_ID}' …")
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

    # Tell the model to always predict in English
    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
        language=LANGUAGE, task=TASK
    )
    model.config.suppress_tokens = []

    # ── Collator & metrics ────────────────────────────────────────────────────
    data_collator   = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
    compute_metrics = compute_metrics_fn(processor)

    # ── Training arguments ────────────────────────────────────────────────────
    training_args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,

        # Steps vs epochs
        max_steps=MAX_STEPS if MAX_STEPS > 0 else -1,
        num_train_epochs=NUM_EPOCHS if MAX_STEPS <= 0 else 1,

        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,

        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="cosine",

        fp16=FP16,

        # Evaluation & saving
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,

        # Generation settings for eval
        predict_with_generate=True,
        generation_max_length=225,

        logging_steps=10,
        report_to="none",         # change to "tensorboard" if you want TB logs
        push_to_hub=False,
    )

    # ── Trainer ───────────────────────────────────────────────────────────────
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=ds["train"],
        eval_dataset=ds["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # ── Train ─────────────────────────────────────────────────────────────────
    print("Starting training …")
    trainer.train()

    # ── Save final model & processor ──────────────────────────────────────────
    print(f"Saving fine-tuned model to '{OUTPUT_DIR}' …")
    trainer.save_model(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)
    print("Training complete ✓")


if __name__ == "__main__":
    main()

Loading preprocessed dataset …
  Train size : 200
  Valid size : 50
Loading model 'openai/whisper-small' …


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Starting training …


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


In [3]:
"""
push_to_hub.py
Push processed Whisper dataset + processor to Hugging Face Hub
"""

from datasets import load_from_disk
from transformers import WhisperProcessor
from huggingface_hub import login

# ── Config ─────────────────────────────────────────────
DATASET_DIR = "./whisper_dataset"
REPO_ID     = "aminulpalash506/whisper-librispeech-processed"  # change this
PRIVATE     = True   # set False if you want public
# ───────────────────────────────────────────────────────

def main():
    # 🔐 Login (paste your HF token)
    login()

    # 📦 Load dataset
    print("Loading dataset …")
    ds = load_from_disk(DATASET_DIR)

    # 🚀 Push dataset
    print("Pushing dataset to Hub …")
    ds.push_to_hub(REPO_ID, private=PRIVATE)

    # 📦 Push processor (tokenizer + feature extractor)
    print("Pushing processor …")
    processor = WhisperProcessor.from_pretrained(DATASET_DIR)
    processor.push_to_hub(REPO_ID)

    print("✅ Done!")
    print(f"https://huggingface.co/datasets/{REPO_ID}")


if __name__ == "__main__":
    main()

Loading dataset …
Pushing dataset to Hub …


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  49%|####9     | 4.61MB / 9.36MB            

Pushing processor …
✅ Done!
https://huggingface.co/datasets/aminulpalash506/whisper-librispeech-processed
